In [1]:
# !pip install torch
# !pip install torchvision
# !pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu129 -U


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os

# Set backend for matplotlib to save plots in non-GUI environments
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [2]:
# ---------------------------------------------------
# Hyperparameters and Configuration
# ---------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
lr = 5e-5  # Learning rate for WGAN
batch_size = 64
image_size = 64
channels_img = 1  # MNIST is grayscale
z_dim = 100  # Latent noise dimension
num_epochs = 50
critic_iterations = 5  # Number of critic updates per generator update
clip_value = 0.01  # Weight clipping value for the critic
num_workers = 4 # Use multiple CPU cores to load data

# ---------------------------------------------------
# Data Loading and Transformation
# ---------------------------------------------------
transforms = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(channels_img)], [0.5 for _ in range(channels_img)]),
])

dataset = datasets.MNIST(root="dataset/", train=True, transform=transforms, download=True)

# Use the optimized DataLoader
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True, # Helps speed up data transfer to GPU
)

In [3]:
# ---------------------------------------------------
# Model Definitions
# ---------------------------------------------------
class Generator(nn.Module):
    def __init__(self, z_dim, channels_img, features_g):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, features_g * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 16, features_g * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 2, channels_img, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

class Critic(nn.Module):
    def __init__(self, channels_img, features_d):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels_img, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 8, 1, 4, 2, 0, bias=False),
            # NO SIGMOID at the end for the Critic
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
# ---------------------------------------------------
# Initialization
# ---------------------------------------------------
gen = Generator(z_dim, channels_img, 64).to(device)
critic = Critic(channels_img, 64).to(device)

opt_gen = optim.RMSprop(gen.parameters(), lr=lr)
opt_critic = optim.RMSprop(critic.parameters(), lr=lr)

# For visualization and plotting
fixed_noise = torch.randn(64, z_dim, 1, 1).to(device)
G_losses = []
D_losses = [] # This will store the Critic's loss

# Create directory to save generated images
os.makedirs("gan_samples_WGAN", exist_ok=True)
print("Starting WGAN Training...")
# ---------------------------------------------------
# Main WGAN Training Loop (Optimized)
# ---------------------------------------------------
for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.to(device)
        current_batch_size = real.shape[0]

        # === Train Critic ===
        for _ in range(critic_iterations):
            noise = torch.randn(current_batch_size, z_dim, 1, 1).to(device)
            fake = gen(noise)
            
            critic_real = critic(real).reshape(-1)
            critic_fake = critic(fake).reshape(-1)
            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))
            
            critic.zero_grad()
            # The retain_graph=True argument is no longer needed
            loss_critic.backward()
            opt_critic.step()

            # Clip critic weights
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # === Train Generator ===
        # A new batch of fake images is generated to update the generator
        # This is the key optimization that removes the need for retain_graph=True
        noise = torch.randn(current_batch_size, z_dim, 1, 1).to(device)
        fake = gen(noise)
        output = critic(fake).reshape(-1)
        loss_gen = -torch.mean(output)
        
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()
        
    # --- End of Epoch ---
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss C: {loss_critic:.4f}, Loss G: {loss_gen:.4f}")

    # Append losses for plotting
    G_losses.append(loss_gen.item())
    D_losses.append(loss_critic.item())
    
    # Save generated images for visualization
    with torch.no_grad():
        fake_samples = gen(fixed_noise)
        from torchvision.utils import save_image
        save_image(fake_samples, f"gan_samples_WGAN/sample_epoch_{epoch+1}.png", normalize=True)

print(G_losses)
print(D_losses)
print("Training finished.")

Starting WGAN Training...


In [ ]:
import matplotlib.pyplot as plt
# ---------------------------------------------------
# Plotting
# ---------------------------------------------------
plt.figure(figsize=(10,5))
plt.title("WGAN: Generator and Critic Loss During Training")
plt.plot(G_losses, label="Generator")
plt.plot(D_losses, label="Critic")   
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.savefig("Wgan_loss_plot.png")
print("Loss plot saved to Wgan_loss_plot.png")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Hyperparameters
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Recommended code block to check CUDA details ---

# Check if CUDA (GPU support) is available
if device == "cuda":
    # Get the number of available GPUs
    gpu_count = torch.cuda.device_count()
    print(f"CUDA is available. Using {gpu_count} GPU(s).")
    
    # Print the name of each GPU
    for i in range(gpu_count):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA not available. Using CPU.")

# --- End of recommended block ---

batch_size = 128
image_size = 64
channels_img = 1 # MNIST is grayscale
z_dim = 100 # Latent dimension (noise)
num_epochs = 50 # As required by the assignment

# Quick fix: Renamed 'transforms' variable to avoid conflict with the module
transform_pipeline = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5 for _ in range(channels_img)], [0.5 for _ in range(channels_img)]),
])

dataset = datasets.MNIST(root="dataset/", train=True, transform=transform_pipeline, download=True)


# Change this:
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# To this (using 4 workers is a safe and effective start):
loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True, # Helps speed up data transfer to the GPU
)
# loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,num_workers=4 )

CUDA is available. Using 1 GPU(s).
  GPU 0: NVIDIA GeForce RTX 5060


In [3]:
class Generator(nn.Module):
    def __init__(self, z_dim, channels_img, features_g):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            # Input: N x z_dim x 1 x 1
            nn.ConvTranspose2d(z_dim, features_g * 16, 4, 1, 0, bias=False), # N x f_g*16 x 4 x 4
            nn.BatchNorm2d(features_g * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 16, features_g * 8, 4, 2, 1, bias=False), # N x f_g*8 x 8 x 8
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False), # N x f_g*4 x 16 x 16
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False), # N x f_g*2 x 32 x 32
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(features_g * 2, channels_img, 4, 2, 1, bias=False), # N x channels_img x 64 x 64
            nn.Tanh() # Output: [-1, 1]
        )

    def forward(self, x):
        return self.net(x)

In [4]:
# (Keep the same architecture, just remove the final activation)
class Critic(nn.Module):
    def __init__(self, channels_img, features_d):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            # Input: N x channels_img x 64 x 64
            nn.Conv2d(channels_img, features_d, 4, 2, 1, bias=False), # N x f_d x 32 x 32
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False), # N x f_d*2 x 16 x 16
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False), # N x f_d*4 x 8 x 8
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False), # N x f_d*8 x 4 x 4
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(features_d * 8, 1, 4, 2, 0, bias=False), # N x 1 x 1 x 1
        )

    def forward(self, x):
        return self.net(x)

In [5]:
lr = 5e-5 # Lower learning rate is common for WGAN
# batch_size, image_size, etc. remain the same
critic_iterations = 5 # Train the critic 5 times for every generator training
clip_value = 0.01 # The value for weight clipping

# Initialize models
gen = Generator(z_dim, channels_img, 64).to(device)
critic = Critic(channels_img, 64).to(device)

# Optimizers --> Use RMSprop
opt_gen = optim.RMSprop(gen.parameters(), lr=lr)
opt_critic = optim.RMSprop(critic.parameters(), lr=lr)

In [6]:
# Create a fixed noise vector to see the progression of the generator
fixed_noise = torch.randn(64, z_dim, 1, 1).to(device) # We'll generate a grid of 8x8=64 images

In [7]:
# Before the training loop starts
G_losses = []
D_losses = []

In [8]:
# Main WGAN training loop
for epoch in range(num_epochs):
    for batch_idx, (real, _) in enumerate(loader):
        real = real.to(device)
        batch_size = real.shape[0]

        # Train Critic: max E[critic(real)] - E[critic(fake)]
        # equivalent to minimizing E[critic(fake)] - E[critic(real)]
        for _ in range(critic_iterations):
            noise = torch.randn(batch_size, z_dim, 1, 1).to(device)
            fake = gen(noise)
            
            critic_real = critic(real).reshape(-1)
            critic_fake = critic(fake).reshape(-1)
            
            loss_critic = -(torch.mean(critic_real) - torch.mean(critic_fake))
            
            critic.zero_grad()
            loss_critic.backward(retain_graph=True) # retain_graph=True because we use fake for generator
            opt_critic.step()

            # Clip critic weights
            for p in critic.parameters():
                p.data.clamp_(-clip_value, clip_value)

        # Train Generator: min -E[critic(fake)] <--> max E[critic(fake)]
        output = critic(fake).reshape(-1)
        loss_gen = -torch.mean(output)
        
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()


    print(f"Epoch [{epoch+1}/{num_epochs}] Loss C: {loss_critic:.4f}, Loss G: {loss_gen:.4f}")

    # You can reuse the visualization code from the standard GAN here!
    # Append the losses for plotting
    G_losses.append(loss_gen.item())
    D_losses.append(loss_critic.item())
    with torch.no_grad():
        # Generate images from the fixed noise vector
        fake_samples = gen(fixed_noise)
        
        # You can save the images or display them
        # Option 1: Save a grid of images
        from torchvision.utils import save_image
        save_image(fake_samples, f"gan_samples_WGAN/sample_epoch_{epoch+1}.png", normalize=True)

Epoch [1/50] Loss C: -1.0048, Loss G: 0.4309
Epoch [2/50] Loss C: -0.3061, Loss G: -0.2162
Epoch [3/50] Loss C: -1.5086, Loss G: 0.7512
Epoch [4/50] Loss C: -1.4038, Loss G: 0.6915
Epoch [5/50] Loss C: -1.5223, Loss G: 0.7563
Epoch [6/50] Loss C: -1.4916, Loss G: 0.7389
Epoch [7/50] Loss C: -1.5273, Loss G: 0.7446
Epoch [8/50] Loss C: -1.4387, Loss G: 0.7097
Epoch [9/50] Loss C: -1.4459, Loss G: 0.7113
Epoch [10/50] Loss C: -1.4918, Loss G: 0.7436
Epoch [11/50] Loss C: -1.4840, Loss G: 0.7393
Epoch [12/50] Loss C: -1.4771, Loss G: 0.7308
Epoch [13/50] Loss C: -1.4237, Loss G: 0.7168
Epoch [14/50] Loss C: -0.7920, Loss G: 0.6217
Epoch [15/50] Loss C: -0.9799, Loss G: 0.5880
Epoch [16/50] Loss C: -0.9359, Loss G: 0.2780
Epoch [17/50] Loss C: -0.9747, Loss G: 0.6047
Epoch [18/50] Loss C: -1.1433, Loss G: 0.6057
Epoch [19/50] Loss C: -1.1344, Loss G: 0.5988
Epoch [20/50] Loss C: -1.4097, Loss G: 0.7000
Epoch [21/50] Loss C: -1.1631, Loss G: 0.5399
Epoch [22/50] Loss C: -1.3342, Loss G: 0.6

In [9]:
# !pip install matplotlib
